In [ ]:

seed = 42
batch_size = 8
patience = 5

target = 'recur'
date = None

save_dir_version = 'v001'



import os
from glob import glob
import random
from tqdm import tqdm
from datetime import datetime

# font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

from src.utils.preprocessing import Preprocessor
from src.models import SupervisedTextDataset, SupervisedPLModel, F1MacroCallback

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from umap import UMAP

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


# 랜덤 시드 설정 (PyTorch Lightning 권장 방식)

# 파이썬 해시 시드
os.environ['PYTHONHASHSEED'] = str(seed)

# 파이썬 random
random.seed(seed)

# numpy, torch, pytorch-lightning는 이미 다른 셀에서 import 되어 있으므로 그대로 사용
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# PyTorch Lightning utility (workers=True로 데이터로더 워커 시드도 설정)
pl.seed_everything(seed, workers=True)

# save_dir_version = 'v002'
# save_dir_prefix = f'CLF-{target}-{inter_fix}'

font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf

font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용

# data_dir_path = sorted(glob(os.path.join('data','reviewed_labels', f'*_{target}.csv')))
data_dir_path = sorted(glob(os.path.join('data','reviewed_labels', f'{target}*')))
print(data_dir_path)

label_dict = {'negative': 0, 'uncertain': 1, 'positive': 2}

In [ ]:
import pandas as pd
import os
from glob import glob

from src.utils.preprocessing import Preprocessor

from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

from collections import Counter
from tqdm import tqdm
import numpy as np


font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용



def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]
measure_dict = dict()

In [ ]:
def save_dir_setup(model_name, prefix='CLF-recur', version='v001', date = None, base_dir = None):
    if base_dir == None:
        base_dir = os.path.join('..', '..', 'logs')

    if date == None:
        log_dir = os.path.join(base_dir, datetime.now().date().isoformat(), '-'.join(filter(None, [prefix, model_name, version])))
    else:
        log_dir = os.path.join(base_dir, date, '-'.join(filter(None, [prefix, model_name, version])))
    os.makedirs(log_dir, exist_ok=True)

    figdir = os.path.join(log_dir,'figures')
    os.makedirs(figdir, exist_ok=True)

    filedir = os.path.join(log_dir,'files')
    os.makedirs(filedir, exist_ok=True)

    return dict(figure=figdir, file=filedir, log=log_dir)


def compute_alpha(df):
    # 클래스별 샘플 수 계산
    class_counts = df['target'].value_counts(sort=False).values
    class_freq = class_counts / class_counts.sum()
    alpha = np.exp(-class_freq / 1.)
    # alpha = (1.0 - class_freq) / (len(class_freq) - 1)
    alpha = alpha / alpha.sum()  # 정규화
    return alpha


def get_model_predictions(pl_model, data_loader):
    """
    Returns logits, targets, and predicted labels for a given model and dataloader.
    """
    pl_model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pl_model.to(device)
    all_logits = []
    all_targets = []
    for batch in tqdm(data_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch.get('attention_mask', None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            
        if 'labels' in batch:
            labels = batch['labels'].to(device)
            all_targets.append(labels.cpu().numpy())
        else:
            labels = None
        with torch.no_grad():
            logits = pl_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        all_logits.append(logits.cpu().numpy())
        
    all_logits = np.concatenate(all_logits)
    all_targets = np.concatenate(all_targets) if len(all_targets) > 0 else None
    all_preds = np.argmax(all_logits, axis=1)
    return dict(
        logits = all_logits, 
        targets = all_targets, 
        preds = all_preds
    )


In [ ]:
df = pd.read_excel(data_dir_path[1])
test_df = pd.read_excel(data_dir_path[0])


df['target'] = df['label'].map(lambda x: label_dict[x] if x in label_dict else 0)
# test_df['target'] = test_df['label'].map(lambda x: label_dict[x] if x in label_dict else 0)
alpha = compute_alpha(df)

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['target'],
    random_state=seed
)


In [ ]:
# model_dict = dict(
#     Bluebert = 'bluebert_pubmed_mimic_uncased_L-12_H-768_A-12',
#     Bio_Clinical_Bert = 'Bio_ClinicalBERT',
#     BioSimCSE_BioLinkBERT_base = 'BioSimCSE-BioLinkBERT-BASE',
#     MS_BiomedNLP_BiomedBERT_base = 'BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
#     ClinicalBERT = 'ClinicalBERT',
#     MedEmbed_base = 'MedEmbed-base-v0.1',
#     pubmedbert = 'PubMedBERT-base-uncased-sts-combined',
#     Multilingual_E5_base = 'multilingual-e5-base',
# )
model_dict = dict(
    recur = 'PubMedBERT-base-uncased-sts-combined',
    metas = 'MedEmbed-base-v0.1',
)
model_dir_path = os.path.join('..','..','..','..','model')


In [ ]:
target = 'metas'
model_path = model_dict[target]

tokenizer = AutoTokenizer.from_pretrained(os.path.join(model_dir_path, model_path))
language_model = AutoModel.from_pretrained(
        os.path.join(model_dir_path, model_path),
        # dtype="bfloat16",
    )
# print(model_name, model_path, )
print('params : ', language_model.num_parameters())
print(language_model.config.hidden_size)


In [ ]:
language_model.encoder

In [ ]:
model_path = model_dict[target]

save_dict = save_dir_setup(
    model_path, 
    prefix=target, 
    version=save_dir_version,
    date = date,
    )

tokenizer = AutoTokenizer.from_pretrained(os.path.join(model_dir_path, model_path))
language_model = AutoModel.from_pretrained(
        os.path.join(model_dir_path, model_path),
        # dtype="bfloat16",
    )
# print(model_name, model_path, )
print('params : ', language_model.num_parameters())
print(language_model.config.hidden_size)

# save_dict = save_dir_setup(model_name, prefix=save_dir_prefix, version=save_dir_version)
# break


# Stratified split (9:1)
# Dataset & DataLoader
train_dataset = SupervisedTextDataset(train_df, tokenizer,text_col = 'prep_text')
val_dataset = SupervisedTextDataset(val_df, tokenizer,text_col = 'prep_text')
test_dataset = SupervisedTextDataset(test_df, tokenizer,text_col = 'prep_text')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Lightning model
pl_model = SupervisedPLModel(
    encoder=language_model,
    hidden_dim=language_model.config.hidden_size,
    num_classes=len(label_dict),
    lr=1e-5,
    gamma=2.0,
    alpha=alpha
)

# Callbacks
early_stop_cb = EarlyStopping(
    monitor='val_f1_macro',
    patience=patience,
    mode='max',
    verbose=True
)
ckpt_cb = ModelCheckpoint(
    dirpath=save_dict['file'],
    filename='clf-{epoch:02d}-{val_f1_macro:.4f}',
    monitor='val_f1_macro',
    mode='max',
    save_top_k=1,
    save_last = True,
    save_weights_only=True
)

# Trainer
trainer = pl.Trainer(
    max_epochs=30,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    callbacks=[early_stop_cb, ckpt_cb, F1MacroCallback(val_loader)],
    # log_every_n_steps=10,
    # fast_dev_run=3
)

trainer.fit(pl_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
# print("Best checkpoint:", ckpt_cb.best_model_path)


In [ ]:

ckpt_path = glob(os.path.join(save_dict['file'], '*epoch*.ckpt'))[0]
pl_model = SupervisedPLModel.load_from_checkpoint(ckpt_path, encoder=language_model)
pl_model.eval()

val_outputs = get_model_predictions(pl_model, val_loader)
test_outputs = get_model_predictions(pl_model, test_loader)

plt.rcParams['font.size'] = 15
label_order = ['Negative', 'Uncertain', 'Positive']
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
sbn.heatmap(confusion_matrix(val_outputs['targets'], val_outputs['preds']), annot=True, fmt='d', cmap='Blues',
            xticklabels=label_order,
            yticklabels=label_order, 
            ax=ax, 
            cbar = False
            )
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
fig.tight_layout()
fig.savefig(os.path.join(save_dict['figure'], 'val_confusion_matrix.png'), dpi = 400)
plt.close()


# fig, ax = plt.subplots(1, 1, figsize=(7, 6))
# sbn.heatmap(confusion_matrix(test_outputs['targets'], test_outputs['preds']), annot=True, fmt='d', cmap='Blues',
#             xticklabels=label_order,
#             yticklabels=label_order, 
#             ax=ax, 
#             cbar = False
#             )
# ax.set_xlabel('Predicted Label')
# ax.set_ylabel('True Label')
# fig.tight_layout()
# fig.savefig(os.path.join(save_dict['figure'], 'test_confusion_matrix.png'), dpi = 400)
# plt.close()


inv_label_dict = {v: k for k, v in label_dict.items()}
test_df['prediction'] = [inv_label_dict[i] for i in test_outputs['preds']]
test_df.to_excel(os.path.join(save_dict['file'], 'test-with-pred.xlsx'), index=False)
# test_df.query('prediction == "positive"').to_excel(os.path.join(save_dict['file'], 'test-pred-positive.xlsx'), index=False)
# test_df.query('prediction == "uncertain"').to_excel(os.path.join(save_dict['file'], 'test-pred-uncertain.xlsx'), index=False)

In [ ]:
save_dict['file']